In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Chargement des données

df_pairs = pd.read_csv('dim_pairs.csv')
df_daily_klines = pd.read_csv('fct_daily_klines.csv')
df_return_correlations = pd.read_csv('fct_return_correlations.csv')
df_pair_overview = pd.read_csv('mart_pair_overview.csv')

In [3]:
# Exploration rapide

print(df_pairs.head())
print(df_daily_klines.head())
print(df_return_correlations.head())
print(df_pair_overview.head())

def check_nan(df):
    for column in df:
        taux_nan = (df[column].isna().sum()/len(df)) * 100
        print(f"df, {column}: {taux_nan} %")

print(check_nan(df_pairs))
print(check_nan(df_daily_klines))
print(check_nan(df_return_correlations))
print(check_nan(df_pair_overview))

    symbol base_asset quote_asset base_name
0   BTCEUR        BTC         EUR   Bitcoin
1  BTCUSDT        BTC        USDT   Bitcoin
2   ETHEUR        ETH         EUR  Ethereum
3  ETHUSDT        ETH        USDT  Ethereum
4   SOLEUR        SOL         EUR    Solana
   symbol  trade_date                       open_time  \
0  BTCEUR  2024-02-16  2024-02-16 00:00:00.000000 UTC   
1  BTCEUR  2024-02-17  2024-02-17 00:00:00.000000 UTC   
2  BTCEUR  2024-02-18  2024-02-18 00:00:00.000000 UTC   
3  BTCEUR  2024-02-19  2024-02-19 00:00:00.000000 UTC   
4  BTCEUR  2024-02-20  2024-02-20 00:00:00.000000 UTC   

                       close_time  open_price  high_price  low_price  \
0  2024-02-16 23:59:59.999000 UTC    48218.28    48968.53   47914.55   
1  2024-02-17 23:59:59.999000 UTC    48428.50    48463.38   46950.00   
2  2024-02-18 23:59:59.999000 UTC    47978.74    48636.66   47529.80   
3  2024-02-19 23:59:59.999000 UTC    48392.53    48724.20   48000.01   
4  2024-02-20 23:59:59.999000 UTC

In [4]:
# 1. Garder une ligne par jour pour la paire choisie
paire = "BTCEUR"
btc = df_daily_klines.loc[df_daily_klines["symbol"] == paire].copy()
btc["trade_date"] = pd.to_datetime(btc["trade_date"])
btc = btc.set_index("trade_date").sort_index()

# 2. Créer des indicateurs connus à la clôture du jour J
btc["ecart_ma_7j"] = btc["close_price"] / btc["ma_7d"] - 1
btc["ecart_ma_30j"] = btc["close_price"] / btc["ma_30d"] - 1
btc["amplitude_jour"] = (btc["high_price"] - btc["low_price"]) / btc["open_price"]
btc["volume_relatif_7j"] = btc["base_volume"] / btc["base_volume"].rolling(7).mean()

features = [
    "daily_return",               # Variation du prix depuis la clôture précédente
    "ecart_ma_7j",                # Écart entre le prix et sa moyenne sur 7 jours
    "ecart_ma_30j",               # Même calcul sur 30 jours
    "amplitude_jour",             # Écart entre le plus haut et le plus bas du jour
    "volume_relatif_7j",          # 1,5 = volume égal à 1,5 fois la moyenne sur 7 jours
    "volatility_30d_annualized",  # Volatilité déjà calculée dans ton CSV
]

In [5]:
# 1. Calculer la variation entre la clôture du jour J et celle du lendemain
rendement_demain = btc["close_price"].shift(-1) / btc["close_price"] - 1

# 2. Créer la cible à prédire
seuil = 0.01  # 1 %
btc["signal"] = "conserver"
btc.loc[rendement_demain > seuil, "signal"] = "achat"
btc.loc[rendement_demain < -seuil, "signal"] = "vente"

# 3. Garder les jours avec 30 rendements d'historique et un lendemain connu
jours_valides = (btc["nb_rendements_30d"] == 30) & rendement_demain.notna()
df_ml = btc.loc[jours_valides, features + ["signal"]].dropna().copy()

# 4. Séparer les indicateurs (X) et la cible (y)
X = df_ml[features]
y = df_ml["signal"]

print(X.shape)
print(y.value_counts())

(983, 6)
signal
conserver    433
vente        280
achat        270
Name: count, dtype: int64


In [6]:
modeles = {
    "Régression logistique": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_leaf_nodes=15, random_state=42
    ),
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    "Référence majoritaire": DummyClassifier(strategy="most_frequent"),
}

# 1. Découper une seule fois pour que les 5 modèles utilisent les mêmes dates
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    shuffle=False
)

# 2. Garder un jour d'écart, car le signal de J utilise la clôture de J+1
X_train = X_train.iloc[:-1]
y_train = y_train.iloc[:-1]

# 3. Entraîner et tester chaque modèle
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    print(nom)
    print(f"  Accuracy train : {modele.score(X_train, y_train):.3f}")
    print(f"  Accuracy test  : {accuracy_score(y_test, y_pred):.3f}")
    print(f"  F1 macro test  : {f1_score(y_test, y_pred, average='macro'):.3f}")

resultats = []
predictions = pd.DataFrame({"signal_reel": y_test})

for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    predictions[nom] = y_pred
    resultats.append({
        "Modèle": nom,
        "Accuracy train": modele.score(X_train, y_train),
        "Accuracy test": accuracy_score(y_test, y_pred),
        "F1 macro test": f1_score(y_test, y_pred, average="macro"),
    })

scores = pd.DataFrame(resultats).set_index("Modèle")
print(scores.sort_values("F1 macro test", ascending=False).round(3))

Régression logistique
  Accuracy train : 0.469
  Accuracy test  : 0.477
  F1 macro test  : 0.283
Arbre de décision
  Accuracy train : 0.499
  Accuracy test  : 0.462
  F1 macro test  : 0.339
Random Forest
  Accuracy train : 0.601
  Accuracy test  : 0.467
  F1 macro test  : 0.312
Gradient Boosting
  Accuracy train : 0.999
  Accuracy test  : 0.345
  F1 macro test  : 0.293
SVM
  Accuracy train : 0.511
  Accuracy test  : 0.472
  F1 macro test  : 0.277
Référence majoritaire
  Accuracy train : 0.427
  Accuracy test  : 0.497
  F1 macro test  : 0.221
                       Accuracy train  Accuracy test  F1 macro test
Modèle                                                             
Arbre de décision               0.499          0.462          0.339
Random Forest                   0.601          0.467          0.312
Gradient Boosting               0.999          0.345          0.293
Régression logistique           0.469          0.477          0.283
SVM                             0.511       

In [7]:
import pandas as pd
from sklearn.metrics import classification_report

resultats_signaux = []

for nom, modele in modeles.items():
    y_pred = modele.predict(X_test)

    rapport = classification_report(
        y_test,
        y_pred,
        labels=["achat", "vente"],
        output_dict=True,
        zero_division=0
    )

    for signal in ["achat", "vente"]:
        resultats_signaux.append({
            "Modèle": nom,
            "Signal": signal,
            "Précision": rapport[signal]["precision"],
            "Rappel": rapport[signal]["recall"],
            "F1": rapport[signal]["f1-score"],
            "Signaux prédits": (y_pred == signal).sum(),
            "Signaux réels": int(rapport[signal]["support"])
        })

scores_signaux = pd.DataFrame(resultats_signaux)
scores_signaux = scores_signaux.sort_values(
    ["Signal", "F1"], ascending=[True, False]
)

display(scores_signaux.round(3))

,Modèle,Signal,Précision,Rappel,F1,Signaux prédits,Signaux réels
2,Arbre de décision,achat,0.278,0.233,0.253,36,43
6,Gradient Boosting,achat,0.208,0.256,0.229,53,43
4,Random Forest,achat,0.243,0.209,0.225,37,43
8,SVM,achat,0.240,0.140,0.176,25,43
0,Régression logistique,achat,0.214,0.070,0.105,14,43
10,Référence majoritaire,achat,0.000,0.000,0.000,0,43
7,Gradient Boosting,vente,0.216,0.143,0.172,37,56
3,Arbre de décision,vente,0.312,0.089,0.139,16,56
1,Régression logistique,vente,0.250,0.054,0.088,12,56
5,Random Forest,vente,0.250,0.036,0.062,8,56


In [8]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
